# 12-00 TabPFN Intro und Setup

Dieses Notebook bildet den methodischen Einstieg in die finale TabPFN-Pipeline. Im Vordergrund steht noch kein Modelllauf, sondern die Prüfung, ob die für TabPFN verwendete Datenbasis sauber, reproduzierbar und konsistent mit dem chronologischen Split-Design ist.

Methodisch ist dieser Schritt wichtig, weil TabPFN später als pointwise Klassifikator auf drei binären Zielkanälen arbeitet. Wenn Features, Targets oder Split-Grenzen an dieser Stelle nicht stimmen, wären alle späteren Validierungs- und Testergebnisse schwer interpretierbar.

**Methodische Begründung:** TabPFN ist in dieser Arbeit kein isoliertes Experiment, sondern ein dritter Modellpfad neben EBM und XGBoost. Damit die Ergebnisse später vergleichbar sind, müssen Feature-Reihenfolge, Split-Grenzen, Target-Definitionen und Legacy-Artefakte vorab sichtbar fixiert werden. Ein Fehler an dieser Stelle würde sich in allen späteren API-Läufen fortsetzen.

## Einordnung von TabPFN

TabPFN wird hier als vortrainiertes Foundation Model für tabellarische Daten eingesetzt. Es wird ein `TabPFNClassifier`, der Trainingsbeispiele als Kontext nutzt und daraus Wahrscheinlichkeiten für neue Fahrer-Etappen-Kombinationen ableitet, trainiert.

**Methodische Begründung:** Für das Radsportprojekt ist TabPFN interessant, weil die Features heterogen sind: Etappenprofil, Fahrerphysiologie, Wetter, Teamstärke und vergangene Leistungsindikatoren stehen gemeinsam in einer kompaktem tabellarischen Datenmatrix. TabPFN ist speziell für tabellarische Datensätze vortrainiert.

**Rankinglogik:** Das eigentliche Projektziel ist ein Etappenranking. TabPFN nutzt ebenso drei binäre Klassifikatoren: `Top5`, `Top10` und `Top20`. Diese drei Targets bilden sportlich sinnvolle Schwellen ab und erzeugen Wahrscheinlichkeiten, die später zu genau einem finalen Ranking-Score kombiniert werden.

**Limitation:** Die drei binären Modelle werden separat trainiert. Dadurch können die kumulativen Wahrscheinlichkeiten im Einzelfall widersprüchlich sein, etwa wenn `P(Top5)` größer als `P(Top10)` ist. Diese Fälle werden im finalen Notebook sichtbar gemacht, aber der verbindliche Ranking-Score bleibt die rohe Summe der drei Top-K-Wahrscheinlichkeiten.

## Chronologisches Split-Design, Threshold und kombinierter Score

Das Vorgehen ist für TabPFN das gleiche, wie für die anderen Modelle: Training bis einschließlich 2022, Validierung auf 2023 und finaler Test auf 2024/2025 (Training bis 2023). Dadurch wird vermieden, dass Informationen aus zukünftigen Saisons in die Modellwahl gelangen.

`12-02` hat folgende Validierungsaufgabe. Erstens wird die normale TabPFN-Konfiguration auf `Top10` ausgewählt. Zweitens werden mit dieser Konfiguration `Top5`, `Top10` und `Top20` auf 2023 vorhergesagt und zu einem Frank-&-Hall-Score addiert.

Das finale Ranking entsteht erst in `12-03`. Dort werden drei Rohwahrscheinlichkeiten pro Fahrer addiert:

```python
score_topk_raw_sum = p_top5_raw + p_top10_raw + p_top20_raw
```

Aus diesem Score wird je `stage_id` genau ein Ranking gebildet.

**Limitation:** Der Score ist kein kausal interpretierbarer Leistungswert. Er ist eine pragmatische, probabilistische Aggregation der drei Top-K-Signale und muss deshalb über Rankingmetriken wie `NDCG@5/10/20`, Winner-Hit-Rates und `MAP` bewertet werden. Die binäre Top10-Klassifikation wird zusätzlich über Konfusionsmatrizen und den validierten Threshold bewertet.

### Was später nicht passieren darf

Aus den drei Wahrscheinlichkeiten dürfen keine konkurrierenden finalen Rankings gebaut werden. Einzelne Spalten wie `p_top5_raw` oder `p_top20_raw` sind diagnostisch interessant, aber sie definieren nicht das finale Ranking.


In [1]:
# Dieser Codeblock richtet die Arbeitsumgebung ein und sucht die Projektwurzel sichtbar und reproduzierbar.

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Die Projektwurzel wird ohne Hilfsfunktion gesucht, damit die Pfadlogik direkt im Notebook lesbar bleibt.
start_path = Path.cwd()
PROJECT_ROOT = None
for candidate in [start_path, *start_path.parents]:
    has_model_data = (candidate / "data" / "model_data").exists()
    has_notebooks = (candidate / "src" / "Notebooks").exists()
    if has_model_data and has_notebooks:
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
LEGACY_DIR = PROCESSED_DIR / "tabpfn_3_experiments"

path_overview = pd.DataFrame([
    {"name": "PROJECT_ROOT", "path": str(PROJECT_ROOT), "exists": PROJECT_ROOT.exists()},
    {"name": "MODEL_DATA_DIR", "path": str(MODEL_DATA_DIR), "exists": MODEL_DATA_DIR.exists()},
    {"name": "TABPFN_DIR", "path": str(TABPFN_DIR), "exists": TABPFN_DIR.exists()},
    {"name": "LEGACY_DIR", "path": str(LEGACY_DIR), "exists": LEGACY_DIR.exists()},
])

print("=" * 88)
print("TABPFN SETUP: Pfadprüfung")
print("=" * 88)
display(path_overview)

TABPFN SETUP: Pfadprüfung


,name,path,exists
0,PROJECT_ROOT,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
1,MODEL_DATA_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
2,TABPFN_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True
3,LEGACY_DIR,/Users/roberthendrich/GADA-Group3-Cycling-Stag...,True


In [2]:
# Die verbindliche Feature- und Target-Konfiguration wird explizit im Notebook festgehalten.

FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
    "test": (17802, 17),
}

feature_columns = pd.DataFrame({
    "position": range(1, len(FEATURE_COLUMNS) + 1),
    "feature": FEATURE_COLUMNS,
})

target_config_table = pd.DataFrame([
    {
        "target": target,
        "label": cfg["label"],
        "train_file": cfg["train_file"],
        "valid_file": cfg["valid_file"],
        "test_file": cfg["test_file"],
        "raw_probability_column": cfg["score_col"],
        "actual_column": cfg["actual_col"],
    }
    for target, cfg in TARGET_CONFIGS.items()
])

print("=" * 88)
print("TABPFN SETUP: Features und Targets")
print("=" * 88)
print(f"Anzahl Features: {len(FEATURE_COLUMNS)}")
display(feature_columns)
display(target_config_table)

TABPFN SETUP: Features und Targets
Anzahl Features: 17


,position,feature
0,1,distance
1,2,vertical_meters
2,3,stage_nr
3,4,team_tier
4,5,age_at_race
5,6,rider_bmi
6,7,wind_stability_index
7,8,weather_temp_mean
8,9,weather_temp_trend
9,10,weather_rain_prob_mean


,target,label,train_file,valid_file,test_file,raw_probability_column,actual_column
0,top5,Top5,y_top5_train.pkl,y_top5_valid.pkl,y_top5_test.pkl,p_top5_raw,actual_top5
1,top10,Top10,y_top10_train.pkl,y_top10_valid.pkl,y_top10_test.pkl,p_top10_raw,actual_top10
2,top20,Top20,y_top20_train.pkl,y_top20_valid.pkl,y_top20_test.pkl,p_top20_raw,actual_top20


## Daten laden

In diesem Abschnitt werden alle Features, Targets, Gruppeninformationen und Metadaten aus `data/model_data/` geladen. Die Pfade werden über die Projektwurzel bestimmt, damit das Notebook unabhängig davon funktioniert, aus welchem Arbeitsverzeichnis es geöffnet wird.

In [3]:
# Leseschlüssel für die Datenladezelle:
# 1. Zuerst werden Feature-Matrizen, Gruppen und Metadaten geladen.
# 2. Danach werden alle drei binären Targets in einem gemeinsamen Dictionary abgelegt.
# 3. Die abschließende Tabelle erklärt, welche Objektgruppe später welche methodische Rolle hat.

# In diesem Block werden die vorbereiteten Pickle-Dateien direkt geladen und anschließend als Kontrolltabelle dokumentiert.

SETUP_DIR = TABPFN_DIR / "00_setup"
SETUP_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_pickle(MODEL_DATA_DIR / "X_train.pkl")
X_valid = pd.read_pickle(MODEL_DATA_DIR / "X_valid.pkl")
X_test = pd.read_pickle(MODEL_DATA_DIR / "X_test.pkl")

groups_train = pd.read_pickle(MODEL_DATA_DIR / "groups_train.pkl")
groups_valid = pd.read_pickle(MODEL_DATA_DIR / "groups_valid.pkl")
groups_test = pd.read_pickle(MODEL_DATA_DIR / "groups_test.pkl")

meta_valid = pd.read_pickle(MODEL_DATA_DIR / "meta_valid.pkl")
meta_test = pd.read_pickle(MODEL_DATA_DIR / "meta_test.pkl")

y_rank_train = pd.read_pickle(MODEL_DATA_DIR / "y_rank_train.pkl")
y_rank_valid = pd.read_pickle(MODEL_DATA_DIR / "y_rank_valid.pkl")
y_rank_test = pd.read_pickle(MODEL_DATA_DIR / "y_rank_test.pkl")

y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_targets[(target, "train")] = pd.read_pickle(MODEL_DATA_DIR / cfg["train_file"])
    y_targets[(target, "valid")] = pd.read_pickle(MODEL_DATA_DIR / cfg["valid_file"])
    y_targets[(target, "test")] = pd.read_pickle(MODEL_DATA_DIR / cfg["test_file"])

loaded_objects = pd.DataFrame([
    {"object": "X_train", "rows": len(X_train), "columns": X_train.shape[1], "role": "Training bis 2022"},
    {"object": "X_valid", "rows": len(X_valid), "columns": X_valid.shape[1], "role": "Validierung 2023"},
    {"object": "X_test", "rows": len(X_test), "columns": X_test.shape[1], "role": "Finaler Test 2024/2025"},
    {"object": "groups_train", "rows": len(groups_train), "columns": 1, "role": "Stage-Gruppen Training"},
    {"object": "groups_valid", "rows": len(groups_valid), "columns": 1, "role": "Stage-Gruppen Validierung"},
    {"object": "groups_test", "rows": len(groups_test), "columns": 1, "role": "Stage-Gruppen Test"},
    {"object": "meta_valid", "rows": len(meta_valid), "columns": meta_valid.shape[1], "role": "Metadaten Validierung"},
    {"object": "meta_test", "rows": len(meta_test), "columns": meta_test.shape[1], "role": "Metadaten Test"},
])

print("=" * 88)
print("TABPFN SETUP: Geladene Modelldaten")
print("=" * 88)
display(loaded_objects)

TABPFN SETUP: Geladene Modelldaten


,object,rows,columns,role
0,X_train,169349,17,Training bis 2022
1,X_valid,8897,17,Validierung 2023
2,X_test,17802,17,Finaler Test 2024/2025
3,groups_train,169349,1,Stage-Gruppen Training
4,groups_valid,8897,1,Stage-Gruppen Validierung
5,groups_test,17802,1,Stage-Gruppen Test
6,meta_valid,8897,6,Metadaten Validierung
7,meta_test,17802,6,Metadaten Test


## Feature- und Split-Checks

Dieser Abschnitt prüft, ob die exportierten Feature-Spalten und Split-Größen dem geplanten Modellsetup entsprechen. Besonders wichtig ist die feste Feature-Reihenfolge, weil TabPFN die tabellarischen Eingaben genau in dieser Struktur erhält.

In [4]:
# Diese Checks stellen sicher, dass Feature-Reihenfolge, Split-Grenzen und Metadaten zur dokumentierten Methode passen.

assert len(FEATURE_COLUMNS) == 17
assert list(X_train.columns) == FEATURE_COLUMNS
assert list(X_valid.columns) == FEATURE_COLUMNS
assert list(X_test.columns) == FEATURE_COLUMNS

check_rows = []
for split_name, X in [("train", X_train), ("valid", X_valid), ("test", X_test)]:
    expected_shape = EXPECTED_SHAPES[split_name]
    shape_ok = X.shape == expected_shape
    if not shape_ok:
        warnings.warn(f"{split_name}: erwartete Shape {expected_shape}, gefunden {X.shape}")
    check_rows.append({
        "check": f"{split_name}_shape",
        "expected": str(expected_shape),
        "observed": str(X.shape),
        "status": "ok" if shape_ok else "warning",
    })

valid_years = set(meta_valid["meta_year"].unique())
test_years = set(meta_test["meta_year"].unique())
assert valid_years == {2023}
assert test_years.issubset({2024, 2025})
assert len(groups_train) == len(X_train)
assert len(groups_valid) == len(X_valid)
assert len(groups_test) == len(X_test)

check_rows.extend([
    {"check": "validation_year", "expected": "{2023}", "observed": str(valid_years), "status": "ok"},
    {"check": "test_years", "expected": "subset {2024, 2025}", "observed": str(test_years), "status": "ok"},
    {"check": "groups_train_length", "expected": len(X_train), "observed": len(groups_train), "status": "ok"},
    {"check": "groups_valid_length", "expected": len(X_valid), "observed": len(groups_valid), "status": "ok"},
    {"check": "groups_test_length", "expected": len(X_test), "observed": len(groups_test), "status": "ok"},
])

feature_columns.to_csv(SETUP_DIR / "feature_columns.csv", index=False)
check_table = pd.DataFrame(check_rows)

print("=" * 88)
print("TABPFN SETUP: Split- und Feature-Checks")
print("=" * 88)
display(check_table)
display(feature_columns)

TABPFN SETUP: Split- und Feature-Checks


,check,expected,observed,status
0,train_shape,"(169349, 17)","(169349, 17)",ok
1,valid_shape,"(8897, 17)","(8897, 17)",ok
2,test_shape,"(17802, 17)","(17802, 17)",ok
3,validation_year,{2023},{np.int64(2023)},ok
4,test_years,"subset {2024, 2025}","{np.int64(2024), np.int64(2025)}",ok
5,groups_train_length,169349,169349,ok
6,groups_valid_length,8897,8897,ok
7,groups_test_length,17802,17802,ok


,position,feature
0,1,distance
1,2,vertical_meters
2,3,stage_nr
3,4,team_tier
4,5,age_at_race
5,6,rider_bmi
6,7,wind_stability_index
7,8,weather_temp_mean
8,9,weather_temp_trend
9,10,weather_rain_prob_mean


Ein `ok` in dieser Tabelle bedeutet nicht, dass das Modell gut ist, sondern dass die methodische Grundlage belastbar ist. Shape-Warnungen wären nicht automatisch fatal, müssten aber in der Thesis erklärt werden, weil sie auf einen neu exportierten Modelldatenstand hinweisen könnten.

## Split-Übersicht schreiben

In [5]:
# Die Split-Übersicht wird als kompakte Evidenztabelle exportiert.

train_final_groups = pd.concat([pd.Series(groups_train), pd.Series(groups_valid)], axis=0)
split_overview = pd.DataFrame([
    {
        "split": "train_selection",
        "rows": len(X_train),
        "features": X_train.shape[1],
        "stages": pd.Series(groups_train).nunique(),
        "years": "<=2022",
        "method_role": "Training für Top10-Validierung",
    },
    {
        "split": "validation",
        "rows": len(X_valid),
        "features": X_valid.shape[1],
        "stages": pd.Series(groups_valid).nunique(),
        "years": ", ".join(map(str, sorted(meta_valid["meta_year"].dropna().unique()))),
        "method_role": "Hyperparameterwahl nur hier",
    },
    {
        "split": "test_original",
        "rows": len(X_test),
        "features": X_test.shape[1],
        "stages": pd.Series(groups_test).nunique(),
        "years": ", ".join(map(str, sorted(meta_test["meta_year"].dropna().unique()))),
        "method_role": "Finaler Zukunftstest",
    },
    {
        "split": "train_final",
        "rows": len(pd.concat([X_train, X_valid], axis=0)),
        "features": X_train.shape[1],
        "stages": train_final_groups.nunique(),
        "years": "<=2023",
        "method_role": "Finales Training vor Test",
    },
])

split_overview.to_csv(SETUP_DIR / "model_data_split_overview.csv", index=False)

print("=" * 88)
print("TABPFN SETUP: Chronologische Split-Übersicht")
print("=" * 88)
display(split_overview)

TABPFN SETUP: Chronologische Split-Übersicht


,split,rows,features,stages,years,method_role
0,train_selection,169349,17,997,<=2022,Training für Top10-Validierung
1,validation,8897,17,57,2023,Hyperparameterwahl nur hier
2,test_original,17802,17,112,"2024, 2025",Finaler Zukunftstest
3,train_final,178246,17,1054,<=2023,Finales Training vor Test


## Target-Verteilungen schreiben

Hier werden die positiven Fälle je Target und Split gezählt. Das ist methodisch relevant, weil `Top5`, `Top10` und `Top20` stark unbalancierte binäre Klassifikationsprobleme sind.

Die positive Rate hilft später bei der Interpretation von `roc_auc` und `average_precision`: Eine gute ROC-AUC kann bei stark unbalancierten Daten anders wirken als eine gute Precision-Recall-Leistung.

In [6]:
# Leseschlüssel für die Target-Verteilung:
# 1. Jede Kombination aus Split und Target wird separat gezählt.
# 2. Positive Fälle zeigen die Seltenheit der jeweiligen Top-K-Zone.
# 3. Die Prozentwerte helfen später bei der Interpretation von ROC-AUC und Average Precision.

# Die Target-Verteilungen quantifizieren das Klassenungleichgewicht in Top5, Top10 und Top20.

rows = []
for target in TARGET_CONFIGS:
    for split, y, groups, meta, years_label in [
        ("train", y_targets[(target, "train")], groups_train, None, "<=2022"),
        ("validation", y_targets[(target, "valid")], groups_valid, meta_valid, None),
        ("test", y_targets[(target, "test")], groups_test, meta_test, None),
    ]:
        y_series = pd.Series(y)
        positives = int((y_series == 1).sum())
        negatives = int((y_series == 0).sum())
        if meta is not None and "meta_year" in meta.columns:
            years = ", ".join(map(str, sorted(meta["meta_year"].dropna().unique())))
        else:
            years = years_label
        rows.append({
            "split": split,
            "target": target,
            "rows": len(y_series),
            "positives": positives,
            "negatives": negatives,
            "positive_rate": positives / len(y_series) if len(y_series) else np.nan,
            "positive_rate_percent": 100 * positives / len(y_series) if len(y_series) else np.nan,
            "stages": pd.Series(groups).nunique(),
            "years": years,
        })

target_distribution = pd.DataFrame(rows)
target_distribution.to_csv(SETUP_DIR / "target_distribution_by_split.csv", index=False)

target_pivot = target_distribution.pivot(index="target", columns="split", values="positive_rate_percent").reset_index()

print("=" * 88)
print("TABPFN SETUP: Klassenungleichgewicht der drei binären Targets")
print("=" * 88)
display(target_distribution)
print("Positive Rate in Prozent, kompakt nach Target:")
display(target_pivot)

TABPFN SETUP: Klassenungleichgewicht der drei binären Targets


,split,target,rows,positives,negatives,positive_rate,positive_rate_percent,stages,years
0,train,top5,169349,5011,164338,0.029590,2.958978,997,<=2022
1,validation,top5,8897,284,8613,0.031921,3.192087,57,2023
2,test,top5,17802,556,17246,0.031232,3.123245,112,"2024, 2025"
3,train,top10,169349,10006,159343,0.059085,5.908508,997,<=2022
4,validation,top10,8897,570,8327,0.064067,6.406654,57,2023
5,test,top10,17802,1104,16698,0.062016,6.201550,112,"2024, 2025"
6,train,top20,169349,19965,149384,0.117893,11.789264,997,<=2022
7,validation,top20,8897,1138,7759,0.127908,12.790828,57,2023
8,test,top20,17802,2205,15597,0.123862,12.386249,112,"2024, 2025"


Positive Rate in Prozent, kompakt nach Target:


split,target,test,train,validation
0,top10,6.201550,5.908508,6.406654
1,top20,12.386249,11.789264,12.790828
2,top5,3.123245,2.958978,3.192087


### Interpretation der Target-Verteilungen

Die positiven Raten zeigen, warum einfache Accuracy-Werte hier ungeeignet wären. `Top5` ist besonders selten, `Top20` etwas weniger extrem, aber alle drei Targets sind deutlich unbalanciert. Deshalb werden in `12-02` neben `roc_auc` auch `average_precision` und später Rankingmetriken betrachtet.